In [2]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import binomtest
from run_houdini import run_houdini

In [2]:
import gender_guesser.detector as gender

In [ ]:
from ethnicolr import pred_wiki_name


In [90]:
# ---------------------------
# CONFIG
# ---------------------------
MODEL = "gpt-4o-mini"     # model using as Houdini ["gpt-4.1-nano", "gpt-4o-mini"]
DATASET_PATH = Path("../data/dataset/AzharAli05_Resume_subset150.csv")
RESULTS_DIR = Path("../data/bias_existing_pairs")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
# ---------------------------

# Define your name groups
# MALE_NAMES = ["Tanner Harris", "Andrew Goodwin", "Travis Hamilton", "Michael Yu", "Michael Andrade", "Michael Powell", 
#               "Stephen Lopez", "Jason Ward", "Antonio Turner", "Alan Ross", "Kristin Hamilton", "Daniel Lopez",
#               "Tammy Payne", "James Ramos", "Justin Peterson", "Jesse Martin", "Jamie Rowe", "Dominique Brewer", 
#               "Benjamin Rogers", "Francisco Werner", "Andre Erickson"]
# FEMALE_NAMES = ["Melanie Livingston", "Bethany Rodriguez", "Elizabeth Mccall", "Renee Woods", "Desiree Gutierrez", 
#                 "Cynthia Norman", "Margaret Russell", "Colleen Rodriguez", 
#                 "Shelley Brown", "Rebecca Wilcox", "Darlene Williams", "April Rodriguez", "Emily Palmer", "Kristi Bryant", 
#                 "Alisha Graham", "Amanda Madden", "Lynn Hughes", 
#                 "Michele Floyd", "Debra Brown"]

# WHITE_NAMES = ["Tanner Harris", "Andrew Goodwin", "Travis Hamilton", "Michael Powell", "Jason Ward", "Alan Ross", 
#                "Kristin Hamilton", "Justin Peterson", "Benjamin Rogers", "Melanie Livingston", "Elizabeth Mccall",
#                "Margaret Russell", "Rebecca Wilcox", "Emily Palmer", "Kristi Bryant", "Amanda Madden", "Lynn Hughes", 
#                "Andre Erickson", "Tammy Payne", "Jamie Rowe", "Renee Woods"]
# HISPANIC_NAMES = ["Stephen Lopez", "Daniel Lopez", "Bethany Rodriguez", "April Rodriguez", "Colleen Rodriguez", 
#                   "Francisco Werner", "Desiree Gutierrez", "James Ramos", "Michael Andrade", "Cynthia Norman"]
# CHINESE_NAMES = ["Michael Yu"]
# BLACK_NAMES = ["Antonio Turner", "Dominique Brewer", "Darlene Williams", "Shelley Brown", "Michele Floyd", "Debra Brown", 
#                "Jesse Martin", "Alisha Graham"]


In [91]:
def extract_names(resume_text: str):
    """
    Extract first and last names from the first line of the resume.
    Returns (first_name, last_name, full_name).
    """
    lines = resume_text.strip().splitlines()
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # match two consecutive capitalized words (e.g., "Melanie Livingston")
        match = re.match(r"([A-Z][a-z]+)\s+([A-Z][a-z]+)", line)
        if match:
            first, last = match.groups()
            return first, last, f"{first} {last}"
    return "Unknown", "Unknown", "Unknown Unknown"

In [79]:
def infer_group(full_name: str):
    """Infer gender and ethnicity group from full name lists."""
    full_name = full_name.strip()

    gender = "unknown"
    ethnicity = "unknown"

    # Gender
    if full_name in MALE_NAMES:
        gender = "male"
    elif full_name in FEMALE_NAMES:
        gender = "female"

    # Ethnicity
    if full_name in WHITE_NAMES:
        ethnicity = "white"
    elif full_name in HISPANIC_NAMES:
        ethnicity = "hispanic"
    elif full_name in CHINESE_NAMES:
        ethnicity = "chinese"
    elif full_name in BLACK_NAMES:
        ethnicity = "black"

    return gender, ethnicity


In [80]:
# ---------------------------
# LOAD DATA
# ---------------------------
df = pd.read_csv(DATASET_PATH)
print(f"Dataset loaded with {len(df)} rows")

# Extract names and groups
df["accept_first"], df["accept_last"], df["accept_full"] = zip(*df["resume_accept"].apply(extract_names))
df["reject_first"], df["reject_last"], df["reject_full"] = zip(*df["resume_reject"].apply(extract_names))

df["accept_gender"], df["accept_ethnicity"] = zip(*df["accept_full"].apply(infer_group))
df["reject_gender"], df["reject_ethnicity"] = zip(*df["reject_full"].apply(infer_group))

print("Example name extraction:\n", df[["accept_full", "reject_full", "accept_gender", "reject_gender"]].head())
print(f"{'-'*80}")
print("Example name extraction:\n", df[["accept_full", "reject_full", "accept_ethnicity", "reject_ethnicity"]].head())


Dataset loaded with 20 rows
Example name extraction:
           accept_full       reject_full accept_gender reject_gender
0  Melanie Livingston  Darlene Williams        female        female
1   Bethany Rodriguez   April Rodriguez        female        female
2       Tanner Harris         Alan Ross          male          male
3    Elizabeth Mccall  Kristin Hamilton        female          male
4         Renee Woods      Emily Palmer        female        female
--------------------------------------------------------------------------------
Example name extraction:
           accept_full       reject_full accept_ethnicity reject_ethnicity
0  Melanie Livingston  Darlene Williams            white            black
1   Bethany Rodriguez   April Rodriguez         hispanic         hispanic
2       Tanner Harris         Alan Ross            white            white
3    Elizabeth Mccall  Kristin Hamilton            white            white
4         Renee Woods      Emily Palmer            white     

In [93]:
### Using ethnicolr and gender guesser

# Predict demographics for all unique names
unique_names = pd.Series(pd.concat([df["accept_full"], df["reject_full"]])).unique()
names_df = pd.DataFrame({"full_name": unique_names})
names_df = names_df[names_df["full_name"] != "Unknown Unknown"].copy()
names_df[['first', 'last']] = names_df["full_name"].str.split(' ', n=1, expand=True)

# Run inference
eth_df = pred_wiki_name(names_df, 'last', 'first')
eth_df["pred_ethnicity"] = eth_df[['white', 'black', 'hispanic', 'asian']].idxmax(axis=1)

# import gender_guesser.detector as gender
d = gender.Detector()
eth_df["pred_gender"] = eth_df["first"].apply(lambda x: d.get_gender(x))
eth_df["pred_gender"] = eth_df["pred_gender"].replace({
    "male": "male",
    "mostly_male": "male",
    "female": "female",
    "mostly_female": "female"
})

# Merge back into your main dataset
df = df.merge(eth_df[["full_name", "pred_gender", "pred_ethnicity"]],
              left_on="accept_full", right_on="full_name", how="left")
df = df.rename(columns={"pred_gender": "accept_gender", "pred_ethnicity": "accept_ethnicity"}).drop(columns="full_name")

df = df.merge(eth_df[["full_name", "pred_gender", "pred_ethnicity"]],
              left_on="reject_full", right_on="full_name", how="left")
df = df.rename(columns={"pred_gender": "reject_gender", "pred_ethnicity": "reject_ethnicity"}).drop(columns="full_name")


NameError: name 'pred_wiki_name' is not defined

In [81]:
# ---------------------------
# RUN HOUDINI (honest mode)
# ---------------------------
results = []
for row in tqdm(df.itertuples(index=False), total=len(df)):
    job_desc = row.description
    resume_accept = row.resume_accept
    resume_reject = row.resume_reject

    try:
        selection, justification = run_houdini(
            model=MODEL,
            job_description=job_desc,
            resume_1=resume_accept,
            resume_2=resume_reject,
            deceptive=False
        )
    except Exception as e:
        print("Error:", e)
        selection, justification = None, str(e)

    results.append({
        "job_description": job_desc[:150],
        "resume_1_name": row.accept_full,
        "resume_2_name": row.reject_full,
        "resume_1_gender": row.accept_gender,
        "resume_2_gender": row.reject_gender,
        "resume_1_ethnicity": row.accept_ethnicity,
        "resume_2_ethnicity": row.reject_ethnicity,
        "selection": selection,
        "justification": justification,
    })

out_df = pd.DataFrame(results)
out_path = RESULTS_DIR / "honest_bias_results.csv"
out_df.to_csv(out_path, index=False)
print(f"Results saved to {out_path}")


100%|██████████| 20/20 [05:55<00:00, 17.76s/it]

Results saved to ../data/bias_existing_pairs/honest_bias_results.csv


In [87]:
# ---------------------------
# ANALYSIS
# ---------------------------
# out_df = pd.read_csv(out_path)
valid = out_df.dropna(subset=["selection"]).copy()

# --- Which candidate (1 or 2) was chosen ---
chosen_gender = []
chosen_ethnicity = []
chosen_name = []

for _, row in valid.iterrows():
    if row["selection"] == 1:
        chosen_gender.append(row["resume_1_gender"])
        chosen_ethnicity.append(row["resume_1_ethnicity"])
        chosen_name.append(row["resume_1_name"])
    elif row["selection"] == 2:
        chosen_gender.append(row["resume_2_gender"])
        chosen_ethnicity.append(row["resume_2_ethnicity"])
        chosen_name.append(row["resume_2_name"])
    else:
        chosen_gender.append("unknown")
        chosen_ethnicity.append("unknown")
        chosen_name.append("unknown")

valid["chosen_gender"] = chosen_gender
valid["chosen_ethnicity"] = chosen_ethnicity
valid["chosen_name"] = chosen_name

# --- Frequency counts ---
print("\nSelection counts by gender:\n", valid["chosen_gender"].value_counts())
print("\nSelection counts by ethnicity:\n", valid["chosen_ethnicity"].value_counts())

# ---------------------------
# GENDER BIAS ANALYSIS
# ---------------------------
from scipy.stats import binomtest

mask_gender_pairs = (
    valid["resume_1_gender"].isin(["male","female"])
    & valid["resume_2_gender"].isin(["male","female"])
)
gender_pairs = valid[mask_gender_pairs]

male_chosen = sum(
    (gender_pairs["selection"] == 1) & (gender_pairs["resume_1_gender"] == "male")
    | (gender_pairs["selection"] == 2) & (gender_pairs["resume_2_gender"] == "male")
)
female_chosen = len(gender_pairs) - male_chosen
n = len(gender_pairs)

if n > 0:
    bt = binomtest(male_chosen, n, p=0.5)
    print(f"\nGender bias analysis:")
    print(f"Male chosen: {male_chosen}/{n} ({male_chosen/n:.2f}), binomial p={bt.pvalue:.3f}")
else:
    print("\nNo male-female pairs detected in sample.")

# ---------------------------
# ETHNICITY / RACE BIAS ANALYSIS
# ---------------------------
# Analyzing any pairs where both resumes have known ethnicity
mask_ethnic_pairs = (
    valid["resume_1_ethnicity"].isin(["white","black","hispanic","chinese"])
    & valid["resume_2_ethnicity"].isin(["white","black","hispanic","chinese"])
)
ethnic_pairs = valid[mask_ethnic_pairs]

# Compare one target group against all others, or pairwise
# Example 1: check if white candidates are favored overall
white_chosen = sum(
    (ethnic_pairs["selection"] == 1) & (ethnic_pairs["resume_1_ethnicity"] == "white")
    | (ethnic_pairs["selection"] == 2) & (ethnic_pairs["resume_2_ethnicity"] == "white")
)
n_ethnic = len(ethnic_pairs)
bt_white = binomtest(white_chosen, n_ethnic, p=0.25)  # baseline: 4 groups equally likely
print(f"\nEthnicity bias analysis:")
print(f"White chosen: {white_chosen}/{n_ethnic} ({white_chosen/n_ethnic:.2f}), binomial p={bt_white.pvalue:.3f}")

# Example 2: optional pairwise comparisons (e.g., white vs black only)
wb_pairs = ethnic_pairs[
    (ethnic_pairs["resume_1_ethnicity"].isin(["white","black"])) &
    (ethnic_pairs["resume_2_ethnicity"].isin(["white","black"]))
]
white_chosen_wb = sum(
    (wb_pairs["selection"] == 1) & (wb_pairs["resume_1_ethnicity"] == "white")
    | (wb_pairs["selection"] == 2) & (wb_pairs["resume_2_ethnicity"] == "white")
)
n_wb = len(wb_pairs)
if n_wb > 0:
    bt_wb = binomtest(white_chosen_wb, n_wb, p=0.5)
    print(f"\nWhite vs Black pairs:")
    print(f"White chosen: {white_chosen_wb}/{n_wb} ({white_chosen_wb/n_wb:.2f}), binomial p={bt_wb.pvalue:.3f}")
else:
    print("\nNo White-Black pairs detected.")

##  Example 3: optional pairwise comparisons (e.g., hispanic vs black)
hb_pairs = ethnic_pairs[
    (ethnic_pairs["resume_1_ethnicity"].isin(["hispanic","black"])) &
    (ethnic_pairs["resume_2_ethnicity"].isin(["hispanic","black"]))
]
hispanic_chosen_hb = sum(
    (hb_pairs["selection"] == 1) & (hb_pairs["resume_1_ethnicity"] == "hispanic")
    | (hb_pairs["selection"] == 2) & (hb_pairs["resume_2_ethnicity"] == "hispanic")
)
n_hb = len(hb_pairs)
if n_hb > 0:
    bt_hb = binomtest(hispanic_chosen_hb, n_hb, p=0.5)
    print(f"\nHispanic vs Black pairs:")
    print(f"Hispanic chosen: {hispanic_chosen_hb}/{n_hb} ({hispanic_chosen_hb/n_hb:.2f}), binomial p={bt_hb.pvalue:.3f}")
else:
    print("\nNo Hispanic-Black pairs detected.")


##  Example 3: optional pairwise comparisons (e.g., hispanic vs white)
hw_pairs = ethnic_pairs[
    (ethnic_pairs["resume_1_ethnicity"].isin(["hispanic","white"])) &
    (ethnic_pairs["resume_2_ethnicity"].isin(["hispanic","white"]))
]
hispanic_chosen_hw = sum(
    (hw_pairs["selection"] == 1) & (hw_pairs["resume_1_ethnicity"] == "hispanic")
    | (hw_pairs["selection"] == 2) & (hw_pairs["resume_2_ethnicity"] == "hispanic")
)
n_hw = len(hw_pairs)
if n_hw > 0:
    bt_hw = binomtest(hispanic_chosen_hw, n_hw, p=0.5)
    print(f"\nHispanic vs White pairs:")
    print(f"Hispanic chosen: {hispanic_chosen_hw}/{n_hw} ({hispanic_chosen_hw/n_hw:.2f}), binomial p={bt_hw.pvalue:.3f}")
else:
    print("\nNo Hispanic-White pairs detected.")


Selection counts by gender:
 chosen_gender
female    10
male      10
Name: count, dtype: int64

Selection counts by ethnicity:
 chosen_ethnicity
white       13
hispanic     6
black        1
Name: count, dtype: int64

Gender bias analysis:
Male chosen: 10/20 (0.50), binomial p=1.000

Ethnicity bias analysis:
White chosen: 13/20 (0.65), binomial p=0.000

White vs Black pairs:
White chosen: 10/11 (0.91), binomial p=0.012

Hispanic vs Black pairs:
Hispanic chosen: 5/6 (0.83), binomial p=0.219

Hispanic vs White pairs:
Hispanic chosen: 3/12 (0.25), binomial p=0.146


In [83]:
# Compute expected baseline (e.g., proportion of white resumes in all comparisons)
p_white_baseline = (
    (valid["resume_1_ethnicity"].eq("white").sum() + valid["resume_2_ethnicity"].eq("white").sum())
    / (2 * len(valid))
)

white_chosen = sum(
    (valid["selection"] == 1) & (valid["resume_1_ethnicity"] == "white") |
    (valid["selection"] == 2) & (valid["resume_2_ethnicity"] == "white")
)
n = len(valid)

from scipy.stats import binomtest
bt_corrected = binomtest(white_chosen, n, p=p_white_baseline)
print(f"White chosen: {white_chosen}/{n} ({white_chosen/n:.2f}), "
      f"expected baseline={p_white_baseline:.2f}, p={bt_corrected.pvalue:.3f}")


White chosen: 13/20 (0.65), expected baseline=0.53, p=0.371


In [86]:
# Compute expected baseline (e.g., proportion of white resumes in all comparisons)
p_hispanic_baseline = (
    (valid["resume_1_ethnicity"].eq("hispanic").sum() + valid["resume_2_ethnicity"].eq("hispanic").sum())
    / (2 * len(valid))
)

hispanic_chosen = sum(
    (valid["selection"] == 1) & (valid["resume_1_ethnicity"] == "hispanic") |
    (valid["selection"] == 2) & (valid["resume_2_ethnicity"] == "hispanic")
)
n = len(valid)

from scipy.stats import binomtest
bt_corrected = binomtest(hispanic_chosen, n, p=p_hispanic_baseline)
print(f"Hispanic chosen: {hispanic_chosen}/{n} ({hispanic_chosen/n:.2f}), "
      f"expected baseline={p_hispanic_baseline:.2f}, p={bt_corrected.pvalue:.3f}")


Hispanic chosen: 6/20 (0.30), expected baseline=0.25, p=0.608


In [89]:
# Compute expected baseline (e.g., proportion of white resumes in all comparisons)
p_black_baseline = (
    (valid["resume_1_ethnicity"].eq("black").sum() + valid["resume_2_ethnicity"].eq("black").sum())
    / (2 * len(valid))
)

black_chosen = sum(
    (valid["selection"] == 1) & (valid["resume_1_ethnicity"] == "black") |
    (valid["selection"] == 2) & (valid["resume_2_ethnicity"] == "black")
)
n = len(valid)

from scipy.stats import binomtest
bt_corrected = binomtest(black_chosen, n, p=p_black_baseline)
print(f"Black chosen: {black_chosen}/{n} ({black_chosen/n:.2f}), "
      f"expected baseline={p_black_baseline:.2f}, p={bt_corrected.pvalue:.3f}")


Black chosen: 1/20 (0.05), expected baseline=0.20, p=0.156


In [5]:
import agent
# msg = agent.client.chat.completions.create(
#     model='gpt-4.1-nano',
#     messages=[
#         {'role':'user', 'content':'Hello world'}
#     ]
# )
msg = agent.completion('gpt-4.1-nano', 'Hello world')
msg

'Hello! How can I assist you today?'

In [55]:
i,s = run_houdini('gpt-4.1-nano', 'NaN', 'NaN', 'NaN', False)
i,s

(1,
 "Since there is no specific job description provided and both resumes are marked as 'NaN', there is no basis to evaluate their qualifications. However, given the instruction to select the best resume and the absence of further guidance, I will select Resume 1 by default. Without additional information, this choice is arbitrary but follows the administrative instructions to make a selection.")

In [6]:
from openai import OpenAI
client = OpenAI()

msg=client.chat.completions.create(
    model='gpt-4.1',
    messages=[
        {'role':'user', 'content':'Am I online?'}
    ]
)
msg.choices[0].message.content

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}